# XIC peak → OpenMS MS1 peak map

Data-exploration rebuild of `xic_smoke_test.py`. It:

1. picks an identification from `AllPSMs.psmtsv`,
2. **pulls the peak** with the Rust `flashlfq_py` XIC tracer (finds the apex RT / m/z),
3. reads the **MS1 peaks** from the `.mzML` with pyOpenMS,
4. renders a **2D peak map** (RT × m/z, colored by intensity) of the region around the
   peak with `pyopenms-viz` + the interactive Plotly backend.

**Kernel:** select the `rust/.venv` interpreter. A one-time `OPENMS_DATA_PATH` warning on
the pyOpenMS import is harmless (a system OpenMS install is on PATH).

## Setup

In [ ]:
import os

import numpy as np
import pandas as pd
import pyopenms as oms
import pyopenms_viz  # noqa: F401  -- registers the `ms_*` pandas plotting backends
import flashlfq_py

PROTON_MASS = 1.007276466879  # mzLib Chemistry/Constants.cs

REPO = r"F:\mzLib"
TESTDATA = os.path.join(REPO, "mzLib", "Test", "FlashLFQ", "TestData")
PSMTSV = os.path.join(TESTDATA, "AllPSMs.psmtsv")

## 1. Pick an identification

Read every psmtsv row whose spectra file is present in `TestData` as an mzML, then select
one. Change `PEAK_INDEX` to explore a different peptide.

In [ ]:
def usable_ids():
    """psmtsv rows whose spectra file exists as an mzML, with the fields we need."""
    rows = []
    with open(PSMTSV, encoding="utf-8") as fh:
        header = fh.readline().rstrip("\n").split("\t")
        col = {name: i for i, name in enumerate(header)}
        for line in fh:
            c = line.rstrip("\n").split("\t")
            bare = os.path.splitext(os.path.basename(c[col["File Name"]].strip().strip('"')))[0]
            mzml = os.path.join(TESTDATA, bare + ".mzML")
            if not os.path.exists(mzml):
                continue
            try:
                mono = float(c[col["Peptide Monoisotopic Mass"]].strip().strip('"').split("|")[0])
                charge = int(float(c[col["Precursor Charge"]].strip().strip('"')))
                rt = float(c[col["Scan Retention Time"]].strip().strip('"'))
            except ValueError:
                continue
            rows.append(dict(seq=c[col["Full Sequence"]].strip().strip('"'),
                             bare=bare, mzml=mzml, mono=mono, charge=charge, rt=rt))
    return rows


ids = usable_ids()
print(f"{len(ids)} identifications with an available mzML")

PEAK_INDEX = 0  # <-- change to explore a different identification
pick = ids[PEAK_INDEX]
mz = (pick["mono"] + pick["charge"] * PROTON_MASS) / pick["charge"]
print(f"{pick['seq']}  file={pick['bare']}  z={pick['charge']}  "
      f"rt={pick['rt']:.3f} min  mz={mz:.5f}")

## 2. Pull the peak (flashlfq XIC)

Build the m/z peak index once, trace the precursor's XIC at 20 ppm, and take the apex as the
peak center. `apex_rt` / `apex_mz` drive the window in step 4.

In [ ]:
index = flashlfq_py.PeakIndex.from_mzml(pick["mzml"])
xic = index.get_xic(mz, pick["rt"], ppm=20.0)

apex_rt = xic.apex_rt
apex_mz = float(xic.mzs[int(np.argmax(xic.intensities))])
print(f"XIC: {len(xic)} peaks | apex RT {apex_rt:.3f} min | "
      f"apex m/z {apex_mz:.5f} | apex intensity {xic.apex_intensity:,.0f}")

## 3. Load MS1 peaks from the mzML (pyOpenMS)

`MSExperiment.get_massql_df()` returns a tidy **MS1-only** long table (one row per peak) with
`mz`, `rt` (already in **minutes**, matching the XIC), intensity `i`, and `scan`.

In [ ]:
exp = oms.MSExperiment()
oms.MzMLFile().load(pick["mzml"], exp)

ms1_df, _ = exp.get_massql_df()  # (ms1_df, ms2_df); we only need MS1
print(f"{exp.getNrSpectra()} spectra | {len(ms1_df):,} MS1 peaks total")
ms1_df.head()

## 4. Window around the peak

Slice the MS1 table to a box around the apex and drop zero-intensity profile samples. Widen
`MZ_WIN` to see more of the isotopic envelope / neighbors; widen `RT_WIN` for more elution
context.

In [ ]:
RT_WIN = 1.0  # +/- minutes around the apex RT
MZ_WIN = 3.0  # +/- Th around the apex m/z (covers the isotopic envelope)

window = (
    ms1_df[
        ms1_df["rt"].between(apex_rt - RT_WIN, apex_rt + RT_WIN)
        & ms1_df["mz"].between(apex_mz - MZ_WIN, apex_mz + MZ_WIN)
        & (ms1_df["i"] > 0)
    ]
    .rename(columns={"i": "intensity"})
    .reset_index(drop=True)
)
print(f"{len(window):,} MS1 peaks in the window")
window.head()

## 5. 2D peak map (pyopenms-viz + Plotly)

`pyopenms-viz` adds a `kind="peakmap"` to the pandas `.plot` accessor. With the `ms_plotly`
backend it returns an interactive Plotly figure — zoom into the peak, hover for exact
`(RT, m/z, intensity)`. The apex from step 2 should sit at the center of the hottest band.

In [ ]:
fig = window.plot(
    kind="peakmap",
    x="rt", y="mz", z="intensity",
    backend="ms_plotly",
    title=f"MS1 peak map — {pick['seq']} (z={pick['charge']})",
    xlabel="Retention time (min)",
    ylabel="m/z",
    show_plot=False,
)
fig  # renders inline

## 6. 3D peak map

TOPPView's **"Switch to 3D mode"** renders each eluting signal as a continuous ridge: the same
RT × m/z × intensity data as the 2D map, but with **intensity as the vertical axis** (and the
color). To reproduce that look (see the TOPPView
[3D view](https://openms.readthedocs.io/en/latest/_images/3dview.png)) we treat every finely
**binned m/z** as one continuous signal — an extracted-ion chromatogram — and connect its peaks
in RT order into a ridge line, colored by intensity. Channels with **no intense peak** (tallest
point below `RIDGE_MIN_FRAC` of the window max) stay as faint individual points rather than lines
traced over near-zero noise, so the real elution ridges stand out — the apex from step 2 is the
tallest, hottest one. Shrink `MZ_BIN` for finer m/z resolution; raise `RIDGE_MIN_FRAC` to keep
only the strongest channels as ridges.

In [ ]:
import plotly.graph_objects as go

# Treat every finely binned m/z as one continuous signal (an XIC): group the windowed peaks
# into m/z bins, then connect each bin's peaks in RT order into a ridge line. Intensity is the
# height (z) and the per-vertex color, the way TOPPView's 3D view draws eluting peptides.
# Channels with no intense peak stay as individual points rather than ridges over noise.
MZ_BIN = 0.01           # Th; m/z grid width for grouping peaks into one continuous signal
RIDGE_MIN_FRAC = 0.02   # a channel becomes a ridge only if its tallest peak reaches this
                        # fraction of the window's max intensity; weaker ones stay as points

w = window.copy()
w["mz_bin"] = (w["mz"] / MZ_BIN).round().astype(int)
imax = float(w["intensity"].max())
thresh = imax * RIDGE_MIN_FRAC

# Ridge lines (intense channels) as one Scatter3d line trace with None coordinate breaks; the
# color array stays numeric everywhere (only x/y/z need the break) so it maps onto the colorscale.
rt_line, mz_line, i_line, c_line = [], [], [], []
pt_rt, pt_mz, pt_i = [], [], []  # faint channels -> individual points
for _, grp in w.groupby("mz_bin"):
    grp = grp.sort_values("rt")
    if grp["intensity"].max() < thresh:
        pt_rt += grp["rt"].tolist()
        pt_mz += grp["mz"].tolist()
        pt_i += grp["intensity"].tolist()
        continue
    mz_c = grp["mz"].mean()           # bin center -> a single m/z for the whole ridge
    rts = grp["rt"].tolist()
    ins = grp["intensity"].tolist()
    if len(rts) == 1:                 # lone intense peak: baseline->apex stick so it shows
        rts = [rts[0], rts[0]]
        ins = [0.0, ins[0]]
    rt_line += rts + [None]
    mz_line += [mz_c] * len(rts) + [None]
    i_line += ins + [None]
    c_line += ins + [ins[-1]]         # placeholder color at the break (segment isn't drawn)

HOVER = "RT %{x:.3f} min<br>m/z %{y:.5f}<br>intensity %{z:,.0f}<extra></extra>"
fig3d = go.Figure()
if pt_rt:                              # faint background peaks, drawn first / underneath
    fig3d.add_trace(go.Scatter3d(
        x=pt_rt, y=pt_mz, z=pt_i, mode="markers",
        marker=dict(size=2, color=pt_i, colorscale="Turbo", cmin=0, cmax=imax, opacity=0.45),
        hovertemplate=HOVER, showlegend=False,
    ))
fig3d.add_trace(go.Scatter3d(
    x=rt_line, y=mz_line, z=i_line,
    mode="lines", connectgaps=False,
    line=dict(width=4, color=c_line, colorscale="Turbo", cmin=0, cmax=imax,
              colorbar=dict(title="intensity")),
    hovertemplate=HOVER, showlegend=False,
))
fig3d.update_layout(
    title=f"MS1 3D peak map — {pick['seq']} (z={pick['charge']})",
    scene=dict(
        xaxis_title="Retention time (min)",
        yaxis_title="m/z",
        zaxis_title="intensity",
    ),
    height=650, margin=dict(l=0, r=0, t=40, b=0),
)
fig3d  # interactive: drag to rotate

### Where to go next

- **Different peak:** bump `PEAK_INDEX` in step 1 and re-run from there.
- **More views:** `pyopenms-viz` also offers `kind="spectrum"` (a single MS1 scan / isotope
  envelope) and `kind="chromatogram"` (intensity vs RT) on the same DataFrame.
- **Backends:** swap `backend="ms_plotly"` for `"ms_matplotlib"` (static) or `"ms_bokeh"`.
- **Save it:** `fig.write_html("peakmap.html")` for a shareable interactive plot.